In [ ]:
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3

doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [2]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [ ]:
TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"

input_features = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 
                'B9',
               "B4_minus_B3",
               "B4_div_B3", 
               "ND_B4_B3", 
               'Temperature', 'DewPoint', 'v10n', 
               'doy_sin', 'doy_cos', 
               'latitude', 'longitude'
               ]

X_train = doc_train_augmented[input_features]
y_train = doc_train_augmented[TARGET]

X_test = doc_test[input_features]
y_test = doc_test[TARGET]

X_train.shape, y_train.shape, X_test.shape, y_test.shape


((2732, 19), (2732,), (171, 19), (171,))

In [4]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import xgboost as xgb
from xgboost import XGBRegressor

#Checking version of xgboost
print("XGBoost version:", xgb.__version__)

XGBoost version: 3.1.2


In [ ]:
model_doc_aug = XGBRegressor(
    tree_method="hist",   # histogram algorithm
    device="cuda",        # GPU
    n_estimators=1800,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    min_child_weight=7,
    colsample_bytree=0.8,
    reg_lambda = 4,
    reg_alpha = 0,
    
    objective='reg:squarederror',
    random_state=42
)



model_doc_aug.fit(X_train, y_train)
y_pred = model_doc_aug.predict(X_test)

print("R² for DOC :", r2_score(y_test, y_pred))


# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

#y_pred_orig = np.expm1(y_pred)
#y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



R² for DOC : 0.7000117904206494
R² for DOC on normal: 0.6219682519591032
Final metrics on NORMAL scale (after inverse log):
R²   : 0.6220
MSE  : 1.1789
RMSE : 1.0858
MAE  : 0.6649


In [ ]:
#HYPERPARAM TUNING WITH OPTUNA 

In [ ]:
#using K fold
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.metrics import make_scorer, r2_score
import optuna
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler

#Checking version of xgboost
print("XGBoost version:", xgb.__version__)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

XGBoost version: 3.1.2


In [ ]:
def objective_cv(trial):
    params = {

        "n_estimators": trial.suggest_int("n_estimators", 1000, 2500),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),

        "min_child_weight": trial.suggest_int("min_child_weight", 3, 12),
        "subsample": trial.suggest_float("subsample", 0.65, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.9),

        

        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 3.0),

        "tree_method": "hist",
        "device": "cuda",
        "objective": "reg:squarederror",
        "random_state": 42
    }

    model = XGBRegressor(**params)

    r2_scorer = make_scorer(r2_score)

    scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring=r2_scorer)
    mean_r2 = np.mean(scores)
    return mean_r2

In [13]:
# 3️⃣ Run Optuna study
# --------------------------
study = optuna.create_study(direction="maximize", study_name = "xgboost_study_cuda", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_cv, n_trials=50, show_progress_bar=True, n_jobs=-1)


[I 2026-01-24 11:34:26,684] A new study created in memory with name: xgboost_study_cuda
Best trial: 3. Best value: 0.850101:   2%|▏         | 1/50 [01:13<59:47, 73.22s/it]

[I 2026-01-24 11:35:39,902] Trial 3 finished with value: 0.8501008253378708 and parameters: {'n_estimators': 1279, 'max_depth': 3, 'learning_rate': 0.0309834329987831, 'min_child_weight': 8, 'subsample': 0.8535419345464512, 'colsample_bytree': 0.6962748690993762, 'reg_lambda': 3.3578703878312783, 'reg_alpha': 2.890835667294872}. Best is trial 3 with value: 0.8501008253378708.


Best trial: 3. Best value: 0.850101:   4%|▍         | 2/50 [01:14<24:40, 30.85s/it]

[I 2026-01-24 11:35:41,090] Trial 7 finished with value: 0.83293081922104 and parameters: {'n_estimators': 1266, 'max_depth': 3, 'learning_rate': 0.021134009156218464, 'min_child_weight': 3, 'subsample': 0.8720043604332199, 'colsample_bytree': 0.7250296838231944, 'reg_lambda': 6.043335530425214, 'reg_alpha': 2.3280718823267903}. Best is trial 3 with value: 0.8501008253378708.


Best trial: 2. Best value: 0.939646:   6%|▌         | 3/50 [01:27<17:38, 22.53s/it]

...

[I 2026-01-24 11:40:54,605] Trial 42 finished with value: 0.9794560939907098 and parameters: {'n_estimators': 1578, 'max_depth': 5, 'learning_rate': 0.07530958220937786, 'min_child_weight': 5, 'subsample': 0.6636151981679363, 'colsample_bytree': 0.8293060052837359, 'reg_lambda': 2.409308901879787, 'reg_alpha': 0.33529031476388216}. Best is trial 42 with value: 0.9794560939907098.


Best trial: 42. Best value: 0.979456:  98%|█████████▊| 49/50 [06:28<00:01,  1.24s/it]

[I 2026-01-24 11:40:55,337] Trial 49 finished with value: 0.9429195222628547 and parameters: {'n_estimators': 1211, 'max_depth': 4, 'learning_rate': 0.01949038325483035, 'min_child_weight': 4, 'subsample': 0.6814051661873285, 'colsample_bytree': 0.8120458671993622, 'reg_lambda': 0.610649238452825, 'reg_alpha': 0.33417844377622746}. Best is trial 42 with value: 0.9794560939907098.
[I 2026-01-24 11:40:55,342] Trial 41 finished with value: 0.9786311530004422 and parameters: {'n_estimators': 1921, 'max_depth': 5, 'learning_rate': 0.07019553748110367, 'min_child_weight': 5, 'subsample': 0.6501999319158198, 'colsample_bytree': 0.8325900380392046, 'reg_lambda': 0.5095350273519883, 'reg_alpha': 0.3617481979562722}. Best is trial 42 with value: 0.9794560939907098.
[I 2026-01-24 11:40:55,448] Trial 46 finished with value: 0.9683733789772234 and parameters: {'n_estimators': 1535, 'max_depth': 5, 'learning_rate': 0.017859243399983824, 'min_child_weight': 6, 'subsample': 0.7088043535881874, 'colsam

Best trial: 42. Best value: 0.979456: 100%|██████████| 50/50 [06:29<00:00,  7.79s/it]

[I 2026-01-24 11:40:56,085] Trial 47 finished with value: 0.973034817876652 and parameters: {'n_estimators': 1565, 'max_depth': 5, 'learning_rate': 0.01974556852649059, 'min_child_weight': 6, 'subsample': 0.7098962530765854, 'colsample_bytree': 0.8580590745881956, 'reg_lambda': 0.5410012513873492, 'reg_alpha': 0.32927825479092687}. Best is trial 42 with value: 0.9794560939907098.


In [14]:
print("Best mean CV R²:", study.best_value)
print("Best hyperparameters:", study.best_params)

Best mean CV R²: 0.9794560939907098
Best hyperparameters: {'n_estimators': 1578, 'max_depth': 5, 'learning_rate': 0.07530958220937786, 'min_child_weight': 5, 'subsample': 0.6636151981679363, 'colsample_bytree': 0.8293060052837359, 'reg_lambda': 2.409308901879787, 'reg_alpha': 0.33529031476388216}


In [ ]:
"""
Best mean CV R²: 0.9794560939907098
Best hyperparameters: {'n_estimators': 1578, 
'max_depth': 5, 
'learning_rate': 0.07530958220937786, 
'min_child_weight': 5, 
'subsample': 0.6636151981679363, 
'colsample_bytree': 0.8293060052837359, 
'reg_lambda': 2.409308901879787, 
'reg_alpha': 0.33529031476388216}
"""


In [20]:
best_params =  {'n_estimators': 1578, 'max_depth': 5, 'learning_rate': 0.07530958220937786, 'min_child_weight': 5, 'subsample': 0.6636151981679363, 'colsample_bytree': 0.8293060052837359, 'reg_lambda': 2.409308901879787, 'reg_alpha': 0.33529031476388216}
print(best_params)

{'n_estimators': 1578, 'max_depth': 5, 'learning_rate': 0.07530958220937786, 'min_child_weight': 5, 'subsample': 0.6636151981679363, 'colsample_bytree': 0.8293060052837359, 'reg_lambda': 2.409308901879787, 'reg_alpha': 0.33529031476388216}


In [ ]:
# 4️⃣ Train final model on all augmented train data with best params
# --------------------------
#best_params = study.best_params
final_model = XGBRegressor(
    **best_params,
    tree_method="hist",
    device="cuda",
    objective="reg:squarederror",
    random_state=42
)

final_model.fit(X_train_scaled, y_train)

y_pred_log = final_model.predict(X_test_scaled)

import numpy as np
from sklearn.metrics import r2_score

y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

print("Final R² (log scale):", r2_score(y_test, y_pred_log))
print("Final R² (real scale):", r2_score(y_test_real, y_pred_real))


# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



Final R² (log scale): 0.7083042378380461
Final R² (real scale): 0.6559290615878791
R² for DOC on normal: 0.6559290615878791
Final metrics on NORMAL scale (after inverse log):
R²   : 0.6559
MSE  : 1.0730
RMSE : 1.0358
MAE  : 0.6331


In [ ]:
#Without meteorology

In [1]:
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3

doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [2]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [3]:
TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"

input_features_noMeteo = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 
                'B9',
               "B4_minus_B3",
               "B4_div_B3", 
               "ND_B4_B3", 
               #'Temperature', 'DewPoint', 'v10n',
               'doy_sin', 'doy_cos', 
               'latitude', 'longitude'
               ]

X_train = doc_train_augmented[input_features_noMeteo]
y_train = doc_train_augmented[TARGET]

X_test = doc_test[input_features_noMeteo]
y_test = doc_test[TARGET]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2732, 16), (2732,), (171, 16), (171,))

In [4]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import xgboost as xgb
from xgboost import XGBRegressor

#Checking version of xgboost
print("XGBoost version:", xgb.__version__)

XGBoost version: 3.1.2


In [ ]:

baseline_doc_NoMeteo = XGBRegressor(
    tree_method="hist",   # histogram algorithm
    device="cuda",        # GPU
    n_estimators=1800,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    min_child_weight=7,
    colsample_bytree=0.8,
    reg_lambda = 4,
    reg_alpha = 0,
    objective='reg:squarederror',
    random_state=42
)


baseline_doc_NoMeteo.fit(X_train, y_train)
y_pred = baseline_doc_NoMeteo.predict(X_test)

print("R² for DOC :", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_orig))

# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")




R² for DOC : 0.6652383944527069
R² for DOC on normal: 0.5485873910975492
R² for DOC on normal: 0.5485873910975492
Final metrics on NORMAL scale (after inverse log):
R²   : 0.5486
MSE  : 1.4077
RMSE : 1.1865
MAE  : 0.6924


In [ ]:
#HYPERPARAM TUNING WITH OPTUNA - FOR NO METEO

In [6]:
#using K fold
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.metrics import make_scorer, r2_score
import optuna
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler

#Checking version of xgboost
print("XGBoost version:", xgb.__version__)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
#X_val_scaled = scaler.transform(X_test)
X_test_scaled = scaler.transform(X_test)

XGBoost version: 3.1.2


d:\Miniconda3\envs\PyTorchVirtualEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#NM stands for No Meteo

def objective_cv_NM(trial):
    params = {

        "n_estimators": trial.suggest_int("n_estimators", 1000, 2500),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),

        "min_child_weight": trial.suggest_int("min_child_weight", 3, 12),
        "subsample": trial.suggest_float("subsample", 0.65, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.9),

        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 3.0),

        "tree_method": "hist",
        "device": "cuda",
        "objective": "reg:squarederror",
        "random_state": 42
    }

    model = XGBRegressor(**params)

    r2_scorer = make_scorer(r2_score)

    scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring=r2_scorer)
    mean_r2 = np.mean(scores)
    return mean_r2

In [ ]:

study = optuna.create_study(direction="maximize", study_name = "xgboost_study_cuda", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_cv_NM, n_trials=50, show_progress_bar=True, n_jobs=-1)


[I 2026-02-07 17:51:10,218] A new study created in memory with name: xgboost_study_cuda
Best trial: 15. Best value: 0.823656:   2%|▏         | 1/50 [00:53<43:55, 53.79s/it]

[I 2026-02-07 17:52:04,008] Trial 15 finished with value: 0.823656017694517 and parameters: {'n_estimators': 1071, 'max_depth': 3, 'learning_rate': 0.027068514854313167, 'min_child_weight': 11, 'subsample': 0.8653715674994691, 'colsample_bytree': 0.668344770945311, 'reg_lambda': 1.6774737880810744, 'reg_alpha': 2.181926069497563}. Best is trial 15 with value: 0.823656017694517.


Best trial: 1. Best value: 0.932045:   6%|▌         | 3/50 [01:08<13:09, 16.81s/it] 

[I 2026-02-07 17:52:18,676] Trial 1 finished with value: 0.9320447182110746 and parameters: {'n_estimators': 1357, 'max_depth': 3, 'learning_rate': 0.061344059322270265, 'min_child_weight': 11, 'subsample': 0.7569186020126597, 'colsample_bytree': 0.6582964593829944, 'reg_lambda': 6.183510764523812, 'reg_alpha': 0.3007447088615681}. Best is trial 1 with value: 0.9320447182110746.
[I 2026-02-07 17:52:18,855] Trial 0 finished with value: 0.8224046210964113 and parameters: {'n_estimators': 1112, 'max_depth': 4, 'learning_rate': 0.010851487433055688, 'min_child_weight': 5, 'subsample': 0.8645940873188804, 'colsample_bytree': 0.6946223767541788, 'reg_lambda': 7.588731287642164, 'reg_alpha': 1.4672523994490585}. Best is trial 1 with value: 0.9320447182110746.


Best trial: 1. Best value: 0.932045:   8%|▊         | 4/50 [01:14<09:34, 12.49s/it]

...

[I 2026-02-07 18:00:57,378] Trial 40 finished with value: 0.9761574891105681 and parameters: {'n_estimators': 2464, 'max_depth': 6, 'learning_rate': 0.07926812203733094, 'min_child_weight': 10, 'subsample': 0.7955541977134012, 'colsample_bytree': 0.8757776276453806, 'reg_lambda': 9.142780131183944, 'reg_alpha': 0.21755542680335033}. Best is trial 26 with value: 0.9778974320037014.


Best trial: 26. Best value: 0.977897:  98%|█████████▊| 49/50 [09:51<00:02,  2.92s/it]

[I 2026-02-07 18:01:01,534] Trial 48 finished with value: 0.9725268240966302 and parameters: {'n_estimators': 2294, 'max_depth': 5, 'learning_rate': 0.0741755568313184, 'min_child_weight': 8, 'subsample': 0.7471776892722741, 'colsample_bytree': 0.8690011716063396, 'reg_lambda': 7.864514305599197, 'reg_alpha': 0.2781143635566358}. Best is trial 26 with value: 0.9778974320037014.


Best trial: 26. Best value: 0.977897: 100%|██████████| 50/50 [09:51<00:00, 11.83s/it]

[I 2026-02-07 18:01:01,879] Trial 49 finished with value: 0.9736211956678552 and parameters: {'n_estimators': 2306, 'max_depth': 5, 'learning_rate': 0.0736809571797117, 'min_child_weight': 10, 'subsample': 0.7473783181180905, 'colsample_bytree': 0.8695106417924644, 'reg_lambda': 7.758499950061928, 'reg_alpha': 0.22043585766428087}. Best is trial 26 with value: 0.9778974320037014.


In [31]:
print("Best mean CV R²:", study.best_value)
print("Best hyperparameters:", study.best_params)

Best mean CV R²: 0.9778974320037014
Best hyperparameters: {'n_estimators': 1816, 'max_depth': 6, 'learning_rate': 0.040432659759585106, 'min_child_weight': 3, 'subsample': 0.6646580644242749, 'colsample_bytree': 0.8957206171905826, 'reg_lambda': 3.6867516659251933, 'reg_alpha': 0.05474003456048959}


In [ ]:
#do not follow this result, follow the one below

best_params = study.best_params

final_model_noNM = XGBRegressor(
    **best_params,
    tree_method="hist",
    device="cuda",
    objective="reg:squarederror",
    random_state=42
)

final_model_noNM.fit(X_train_scaled, y_train)

y_pred_log = final_model_noNM.predict(X_test_scaled)

import numpy as np
from sklearn.metrics import r2_score

y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

print("Final R² (log scale):", r2_score(y_test, y_pred_log))
print("Final R² (real scale):", r2_score(y_test_real, y_pred_real))

# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



Final R² (log scale): 0.6402587253824688
Final R² (real scale): 0.5296364550359471
R² for DOC on normal: 0.5296364550359471
Final metrics on NORMAL scale (after inverse log):
R²   : 0.5296
MSE  : 1.4668
RMSE : 1.2111
MAE  : 0.7082


In [ ]:
#follow this result, not the above one

best_params = {'n_estimators': 1816, 'max_depth': 6, 'learning_rate': 0.040432659759585106, 'min_child_weight': 3, 'subsample': 0.6646580644242749, 'colsample_bytree': 0.8957206171905826, 'reg_lambda': 3.6867516659251933, 'reg_alpha': 0.05474003456048959}

final_model_noNM = XGBRegressor(
    **best_params,
    tree_method="hist",
    device="cuda",
    objective="reg:squarederror",
    random_state=42
)

final_model_noNM.fit(X_train_scaled, y_train)

y_pred_log = final_model_noNM.predict(X_test_scaled)

import numpy as np
from sklearn.metrics import r2_score

y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

print("Final R² (log scale):", r2_score(y_test, y_pred_log))
print("Final R² (real scale):", r2_score(y_test_real, y_pred_real))

# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

Final R² (log scale): 0.6511543145523659
Final R² (real scale): 0.5499179803223253
R² for DOC on normal: 0.5499179803223253
Final metrics on NORMAL scale (after inverse log):
R²   : 0.5499
MSE  : 1.4036
RMSE : 1.1847
MAE  : 0.6922


d:\Miniconda3\envs\PyTorchVirtualEnv\Lib\site-packages\xgboost\core.py:750: UserWarning: [00:58:29] WARNING: D:\bld\xgboost-split_1765326833453\work\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [ ]:
#Completed 

# #below here is rough work || Consider it as Miscellaneous work. They are not the part of the above code

In [1]:
from datetime import datetime 
import pandas as pd

csv = "D:\Remote Sensing, ERA5, and DOCgit\Stage_2 Statistical Analysis\StatisticsAnalysis\EmptyRowRemovedData\DOC.csv"
df = pd.read_csv(csv)

date_array = df['sample_date'].to_numpy()
date_array

array(['2018-06-18 15:30:00', '2020-06-22 14:50:00',
       '2021-09-20 13:50:00', '2022-11-14 16:55:00',
       '2022-12-19 17:00:00', '2023-07-17 14:15:00',
       '2017-12-20 17:00:00', '2021-03-24 19:25:00',
       '2021-10-20 17:00:00', '2024-02-22 15:55:00',
       '2017-12-20 16:00:00', '2019-02-18 19:40:00',
       '2021-03-24 17:10:00', '2021-10-20 15:20:00',
       '2024-02-22 19:10:00', '2017-10-16 16:30:00',
       '2018-05-14 16:00:00', '2018-06-18 15:30:00',
       '2019-10-21 16:00:00', '2021-08-16 15:00:00',
       '2022-04-18 14:30:00', '2022-11-14 16:00:00',
       '2019-10-21 17:30:00', '2021-08-16 17:00:00',
       '2022-04-18 17:00:00', '2022-11-14 18:00:00',
       '2017-06-21 16:00:00', '2017-12-20 17:00:00',
       '2018-02-21 17:00:00', '2018-03-28 16:00:00',
       '2018-09-19 16:00:00', '2019-04-17 16:00:00',
       '2020-01-27 19:00:00', '2020-02-26 16:00:00',
       '2020-06-17 16:00:00', '2020-07-15 15:40:00',
       '2022-01-26 16:00:00', '2022-05-18 15:0

In [ ]:
print(max(date_array))
print(min(date_array))

2024-07-02 18:30:00
2017-05-03 15:08:00
